# Attribution Benchmark — Unified Pipeline

> SIF / IF-diag / DVF vs LOO Ground Truth → Spearman + Deletion curve + stat tests.

In [1]:
%run ./functions.ipynb

import time
from functools import lru_cache
from pathlib import Path

def _find_root() -> Path:
    c = Path.cwd().resolve()
    for p in [c, *c.parents]:
        if (p / "notebooks").exists() and (p / "requirements.txt").exists():
            return p
    return c

REPO_ROOT = _find_root()
seed_everything(42)
print(f"Device: {DEVICE}")

Device: cuda


## 1. Config & Load Data

In [ ]:
# ---------- adjustable ----------
BENCH_MODELS = ["MLP", "NCF"]  # VAE needs full matrix, not supported here
ATTR_METHODS = ["sif", "if_diag", "dvf" ]    # "sif", "if_diag", "dvf"
BENCH_SEEDS = [7, 42, 91]

N_CONTROL = 128          # LOO control set size
N_VAL = 512             # validation samples for gradient
LOO_EPOCHS = 10         # retraining epochs per LOO
LOO_LR = 5e-4           # retraining LR
LOO_SEEDS = 3           # seed averaging for utility

# ---------- paths ----------
OUT_DIR = REPO_ROOT / "log" / "notebook_runs"
STEP2_ROOT = OUT_DIR / "step2_training"
BENCH_DIR = OUT_DIR / "attribution_benchmark"
BENCH_DIR.mkdir(parents=True, exist_ok=True)

def _resolve_step2() -> Path:
    lp = STEP2_ROOT / "latest_run.txt"
    if lp.exists():
        ts = lp.read_text(encoding="utf-8").strip()
        p = STEP2_ROOT / ts
        if p.exists(): return p
    cands = sorted([p for p in STEP2_ROOT.iterdir() if p.is_dir()], key=lambda p: p.name, reverse=True)
    if cands: return cands[0]
    raise FileNotFoundError("No step2 run found")

STEP2_DIR = _resolve_step2()
print(f"Step2: {STEP2_DIR}")
print(f"Models: {BENCH_MODELS}  Methods: {ATTR_METHODS}  Seeds: {BENCH_SEEDS}")

Step2: H:\RecAcc\log\notebook_runs\step2_training\20260724_180627
Models: ['MLP', 'NCF']  Methods: ['sif', 'if_diag', 'dvf']  Seeds: [7, 42, 91]


In [3]:
# Load cache + checkpoints
cache = torch.load(REPO_ROOT / "log" / "notebook_cache" / "prepared_data.pt", map_location="cpu", weights_only=False)
matrix = cache["user_item_matrix"]
prepared = PreparedData(
    user_item_matrix=matrix, labels=cache["labels"], sample_ids=cache["sample_ids"],
    user_ids=cache["user_ids"], item_ids=cache["item_ids"],
    num_items=cache["num_items"], num_users=cache["num_users"],
)
NUM_ITEMS = prepared.num_items
print(f"Data: {prepared.num_users} users, {NUM_ITEMS} items, {len(prepared.labels):,} samples")

def _load_ckpts():
    ac = STEP2_DIR / "all_checkpoints.pt"
    if ac.exists():
        raw = torch.load(ac, map_location="cpu", weights_only=False)
        if isinstance(raw, dict):
            return {k.upper(): v if isinstance(v,list) else [] for k,v in raw.items()}
    ck = {}
    for m in ["MLP","NCF","VAE"]:
        f = STEP2_DIR / f"{m.lower()}_checkpoints"
        if f.exists(): ck[m] = [str(p) for p in sorted(f.glob("epoch*.pt"))]
    return ck

CHECKPOINTS = _load_ckpts()
print(f"Checkpoints: { {k:len(v) for k,v in CHECKPOINTS.items()} }")

Data: 19154 users, 8761 items, 775,004 samples
Checkpoints: {'MLP': 7, 'NCF': 12, 'VAE': 38}


## 2. Helpers

In [4]:
def _sub(p, idxs):
    idxs = torch.tensor(list(idxs), dtype=torch.long)
    return PreparedData(
        user_item_matrix=p.user_item_matrix, labels=p.labels[idxs],
        sample_ids=p.sample_ids[idxs], user_ids=p.user_ids[idxs],
        item_ids=p.item_ids[idxs], num_items=p.num_items, num_users=p.num_users,
    )

def _loader_from(p, bs=16, shuffle=False):
    return DataLoader(RecDataset(p), batch_size=bs, shuffle=shuffle)

def _collect(loader, n):
    ds = loader.dataset
    p = PreparedData(
        user_item_matrix=ds.matrix,
        labels=ds.labels, sample_ids=ds.sids,
        user_ids=ds.user_ids, item_ids=ds.item_ids,
        num_items=ds.matrix.shape[1], num_users=ds.matrix.shape[0],
    )
    idx = torch.randperm(len(p.labels))[:min(n, len(p.labels))]
    return _sub(p, idx)

def _spearman(a, b):
    ra = pd.Series(a).rank(method="average").to_numpy()
    rb = pd.Series(b).rank(method="average").to_numpy()
    if np.std(ra) == 0 or np.std(rb) == 0: return 0.0
    return float(np.corrcoef(ra, rb)[0, 1])

def _stable_seed(idxs, rep, seed, model_name):
    key = sorted([int(x) for x in idxs])
    acc = sum((t+1)*(x+17) for t,x in enumerate(key)) % 2147483647
    name_acc = sum(ord(c) for c in model_name)
    return int((seed*1000003 + acc*1315423911 + rep*2654435761 + name_acc) % 2147483647)

## 3. Main Loop — Multi-Seed LOO + Attribution

In [5]:
gt_cache_dir = BENCH_DIR / "loo_gt_cache"
gt_cache_dir.mkdir(parents=True, exist_ok=True)

all_rows = []

for bench_seed in BENCH_SEEDS:
    seed_everything(bench_seed)

    # ---- train/val split ----
    n = prepared.labels.shape[0]
    g = torch.Generator().manual_seed(bench_seed)
    perm = torch.randperm(n, generator=g)
    n_val_samp = int(n * 0.2)

    train_small = _collect(_loader_from(_sub(prepared, perm[n_val_samp:]), 256, False), N_CONTROL)
    val_small   = _collect(_loader_from(_sub(prepared, perm[:n_val_samp]), 256, False), N_VAL)

    ts_loader = _loader_from(train_small, 16, False)
    vs_loader = _loader_from(val_small, 32, False)

    control_ids = [int(x.item()) for x in train_small.sample_ids]
    ctrl_idx = list(range(len(control_ids)))
    ids_set = set(control_ids)

    for model_name in BENCH_MODELS:
        if model_name not in CHECKPOINTS: continue
        print(f"\n{'='*50}")
        print(f"seed={bench_seed}  model={model_name}")

        # ---- load model ----
        m = build_model(name=model_name, num_items=NUM_ITEMS)
        sp = STEP2_DIR / f"{model_name.lower()}_final_state.pt"
        m.load_state_dict(torch.load(sp, map_location="cpu", weights_only=True))
        m.to(DEVICE)
        base_state = {k: v.detach().cpu().clone() for k,v in m.state_dict().items()}

        # ---- utility ----
        @lru_cache(maxsize=4096)
        def utility(tidxs):
            idxs = list(tidxs)
            if len(idxs) == 0: return 0.5
            sub = _sub(train_small, idxs)
            vals = []
            for rep in range(LOO_SEEDS):
                s2 = _stable_seed(idxs, rep, bench_seed, model_name)
                seed_everything(s2)
                u = build_model(name=model_name, num_items=NUM_ITEMS)
                u.load_state_dict(base_state); u.to(DEVICE)
                h, _ = train_model(u, _loader_from(sub, 16, False), vs_loader,
                                    epochs=LOO_EPOCHS, lr=LOO_LR, device=DEVICE,
                                    loss_fn="bpr", verbose=False)
                vals.append(float(evaluate_model(u, vs_loader, device=DEVICE)["auc"]))
            return float(np.mean(vals))

        full_u = utility(tuple(sorted(ctrl_idx)))

        # ---- LOO ground truth ----
        gt_file = gt_cache_dir / f"gt_{model_name}_seed{bench_seed}_n{N_CONTROL}.pt"
        if gt_file.exists():
            gt_data = torch.load(gt_file, map_location="cpu")
            gt = {int(k): float(v) for k,v in gt_data.items() if k != "full_u"}
            print(f"  LOO: cache hit ({len(gt)})")
        else:
            print(f"  LOO: computing {N_CONTROL} samples...")
            gt = {}
            t0 = time.perf_counter()
            for i, sid in enumerate(control_ids):
                keep = tuple(sorted(j for j in ctrl_idx if j != i))
                gt[sid] = full_u - utility(keep)
                if (i+1) % 16 == 0:
                    e = time.perf_counter() - t0
                    print(f"    {i+1}/{N_CONTROL}  ({e:.0f}s, ~{e/(i+1)*(N_CONTROL-i-1):.0f}s left)")
            torch.save({"full_u": full_u, **{str(k):v for k,v in gt.items()}}, gt_file)

        gt_vec = [gt.get(sid, 0.0) for sid in control_ids]

        # ---- SIF ----
        sif_scores = {}
        if "sif" in ATTR_METHODS:
            t0 = time.perf_counter()
            sif_scores = compute_sif(m, vs_loader, ts_loader, max_samples=N_CONTROL, device=DEVICE)
            v = [sif_scores.get(sid, 0.0) for sid in control_ids]
            all_rows.append({"seed": bench_seed, "model": model_name, "method": "SIF",
                            "spearman": _spearman(gt_vec, v),
                            "time": round(time.perf_counter()-t0, 1)})
            print(f"  SIF        ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

        # ---- IF-diag ----
        ifd_scores = {}
        if "if_diag" in ATTR_METHODS:
            t0 = time.perf_counter()
            ifd_scores = compute_if_diag(m, vs_loader, ts_loader, ids_set, device=DEVICE)
            v = [ifd_scores.get(sid, 0.0) for sid in control_ids]
            all_rows.append({"seed": bench_seed, "model": model_name, "method": "IF-diag",
                            "spearman": _spearman(gt_vec, v),
                            "time": round(time.perf_counter()-t0, 1)})
            print(f"  IF-diag    ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

        # ---- DVF ----
        dvf_scores = {}
        if "dvf" in ATTR_METHODS and CHECKPOINTS.get(model_name, []):
            t0 = time.perf_counter()
            def _builder():
                return build_model(name=model_name, num_items=NUM_ITEMS)
            dvf_total, _, _ = compute_dvf_stage(_builder, CHECKPOINTS[model_name],
                                                 ts_loader, vs_loader,
                                                 max_samples=N_CONTROL, device=DEVICE)
            dvf_scores = dvf_total
            v = [dvf_scores.get(sid, 0.0) for sid in control_ids]
            all_rows.append({"seed": bench_seed, "model": model_name, "method": "DVF",
                            "spearman": _spearman(gt_vec, v),
                            "time": round(time.perf_counter()-t0, 1)})
            print(f"  DVF        ρ={all_rows[-1]['spearman']:+.4f}  {all_rows[-1]['time']:.0f}s")

        # ---- Random ----
        rv = []
        rng = np.random.default_rng(bench_seed)
        for _ in range(10):
            rs = {sid: float(len(control_ids)-i) for i,sid in enumerate(rng.permutation(control_ids))}
            rv.append(_spearman(gt_vec, [rs.get(sid,0.0) for sid in control_ids]))
        all_rows.append({"seed": bench_seed, "model": model_name, "method": "Random",
                        "spearman": float(np.median(rv)), "time": 0.0})
        print(f"  Random     ρ={all_rows[-1]['spearman']:+.4f}")

print(f"\nDone — {len(all_rows)} total rows")


seed=7  model=MLP


C:\Users\vanvan\AppData\Local\Temp\ipykernel_57516\1223006325.py:60: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  gt_data = torch.load(gt_file, map_location="cpu")


  LOO: cache hit (64)
  SIF        ρ=+0.1917  0s
  IF-diag    ρ=-0.1358  0s
  DVF        ρ=+0.2679  3s
  Random     ρ=-0.0046

seed=7  model=NCF
  LOO: cache hit (64)
  SIF        ρ=+0.0764  1s
  IF-diag    ρ=+0.0615  1s
  DVF        ρ=+0.0599  7s
  Random     ρ=+0.0961

seed=42  model=MLP
  LOO: computing 64 samples...
    16/64  (25s, ~76s left)
    32/64  (50s, ~50s left)
    48/64  (76s, ~25s left)
    64/64  (101s, ~0s left)
  SIF        ρ=-0.3038  0s
  IF-diag    ρ=+0.3943  0s
  DVF        ρ=-0.2683  3s
  Random     ρ=+0.0221

seed=42  model=NCF
  LOO: computing 64 samples...
    16/64  (32s, ~97s left)
    32/64  (64s, ~64s left)
    48/64  (96s, ~32s left)
    64/64  (127s, ~0s left)
  SIF        ρ=+0.3089  1s
  IF-diag    ρ=-0.0686  1s
  DVF        ρ=-0.1679  8s
  Random     ρ=-0.0139

seed=91  model=MLP
  LOO: computing 64 samples...
    16/64  (27s, ~80s left)
    32/64  (52s, ~52s left)
    48/64  (79s, ~26s left)
    64/64  (104s, ~0s left)
  SIF        ρ=-0.0123  0s
  IF-

## 4. Summary & Tiers

In [6]:
detail_df = pd.DataFrame(all_rows)

agg = detail_df.groupby(["model","method"], as_index=False).agg(
    spearman_mean=("spearman", "mean"),
    spearman_std=("spearman", "std"),
    time_mean=("time", "mean"),
    n_runs=("seed", "count"),
).sort_values(["model","spearman_mean"], ascending=[True, False])

save_table(agg, str(BENCH_DIR / "attribution_summary.csv"))
save_table(detail_df, str(BENCH_DIR / "attribution_detail.csv"))

print("\nAttribution Benchmark Summary:")
print("="*70)
for _, r in agg.iterrows():
    m, s = r['spearman_mean'], r['spearman_std']
    tier = 'STRONG' if m>0.15 and s<0.1 else 'MODERATE' if m>0.05 else 'WEAK' if m>0 else 'INVALID'
    bar = '█' * max(1, int((m+0.1)*20))
    print(f"  {r['model']:5s} {r['method']:10s}  ρ={m:+.4f}±{s:.4f}  {tier:8s}  {bar}")

print(f"\nSaved to: {BENCH_DIR}")
agg

saved: H:\RecAcc\log\notebook_runs\attribution_benchmark\attribution_summary.csv
saved: H:\RecAcc\log\notebook_runs\attribution_benchmark\attribution_detail.csv

Attribution Benchmark Summary:
  MLP   IF-diag     ρ=+0.0936±0.2721  MODERATE  ███
  MLP   DVF         ρ=+0.0193±0.2702  WEAK      ██
  MLP   Random      ρ=+0.0084±0.0134  WEAK      ██
  MLP   SIF         ρ=-0.0415±0.2490  INVALID   █
  NCF   SIF         ρ=+0.1782±0.1189  MODERATE  █████
  NCF   Random      ρ=+0.0162±0.0699  WEAK      ██
  NCF   DVF         ρ=-0.0111±0.1360  INVALID   █
  NCF   IF-diag     ρ=-0.0384±0.0887  INVALID   █

Saved to: H:\RecAcc\log\notebook_runs\attribution_benchmark


,model,method,spearman_mean,spearman_std,time_mean,n_runs
1,MLP,IF-diag,0.093645,0.272148,0.466667,3
0,MLP,DVF,0.019307,0.270226,3.033333,3
2,MLP,Random,0.008369,0.013362,0.000000,3
3,MLP,SIF,-0.041472,0.249005,0.400000,3
7,NCF,SIF,0.178204,0.118924,0.600000,3
6,NCF,Random,0.016194,0.069902,0.000000,3
4,NCF,DVF,-0.011082,0.136002,7.566667,3
5,NCF,IF-diag,-0.038376,0.088659,0.766667,3
